<a href="https://colab.research.google.com/github/gabimazzo-coder/Entrega-Final/blob/main/Entrega_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

PRIMERA PARTE

In [ ]:
import os
from google.colab import userdata

# Leer la clave guardada en los secretos de Colab
API_KEY = userdata.get("API_KEY")


INGESTA y PRUEBA DE API

In [ ]:
import os
import io
from pathlib import Path
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("API_KEY", "")
API_BASE_URL = os.getenv("API_BASE_URL", "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality")

def extraer_dataset_vinos(guardar_local: bool = True) -> pd.DataFrame:
    """
    Descarga e ingesta los datasets completos de vinos tintos y blancos.
    Opcionalmente guarda una copia cruda en data/raw/ para trazabilidad y auditoría.
    """
    url_red = f"{API_BASE_URL}/winequality-red.csv"
    url_white = f"{API_BASE_URL}/winequality-white.csv"

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
    }
    if API_KEY:
        headers["x-api-key"] = API_KEY

    # 1. Descarga vino tinto
    resp_red = requests.get(url_red, headers=headers, timeout=15)
    resp_red.raise_for_status()
    df_red = pd.read_csv(io.StringIO(resp_red.text), sep=";")
    df_red["tipo"] = "tinto"

    # 2. Descarga vino blanco
    resp_white = requests.get(url_white, headers=headers, timeout=15)
    resp_white.raise_for_status()
    df_white = pd.read_csv(io.StringIO(resp_white.text), sep=";")
    df_white["tipo"] = "blanco"

    # 3. Consolidación y normalización de nombres de columnas
    df_total = pd.concat([df_red, df_white], ignore_index=True)
    df_total.columns = [c.strip().lower().replace(" ", "_") for c in df_total.columns]

    # 4. Guardado del dataset crudo consolidado
    if guardar_local:
        carpeta_raw = Path("data/raw")
        carpeta_raw.mkdir(parents=True, exist_ok=True)  # Crea data/raw si no existe
        ruta_archivo = carpeta_raw / "vinos_crudos.csv"
        df_total.to_csv(ruta_archivo, index=False)
        print(f"Dataset crudo guardado en: {ruta_archivo}")

    return df_total

if __name__ == "__main__":
    df = extraer_dataset_vinos()
    print(f"Ingesta exitosa: {len(df)} registros obtenidos.")


Dataset crudo guardado en: data/raw/vinos_crudos.csv
Ingesta exitosa: 6497 registros obtenidos.


2. Pipeline de procesamiento con Polars y DuckDB

In [ ]:
import polars as pl
import duckdb
import pandas as pd

def procesar_con_polars(df_pandas: pd.DataFrame) -> pl.DataFrame:
    """Convierte a Polars, limpia nulos, castea tipos y genera variables derivadas."""
    # Convertir a DataFrame de Polars
    df_pl = pl.from_pandas(df_pandas)

    # Transformaciones analíticas con Polars
    df_procesado = df_pl.with_columns([
        # Asegurar tipos numéricos consistentes
        pl.col("alcohol").cast(pl.Float64),
        pl.col("quality").cast(pl.Int64),
        # Variable objetivo para clasificación (por ej: vino de alta calidad >= 7)
        (pl.col("quality") >= 7).cast(pl.Int64).alias("es_buena_calidad"),
        # Categorización o estandarización
        pl.col("tipo").cast(pl.Categorical)
    ]).drop_nulls()

    return df_procesado

def transformar_con_duckdb(df_polars: pl.DataFrame) -> pl.DataFrame:
    """Aplica agregaciones y transformaciones SQL eficientes usando DuckDB sobre Polars."""
    # DuckDB puede consultar directamente DataFrames de Polars en memoria sin copia
    query = """
    SELECT
        *,
        AVG(alcohol) OVER (PARTITION BY tipo) AS promedio_alcohol_por_tipo,
        fixed_acidity / NULLIF(volatile_acidity, 0) AS ratio_acidez
    FROM df_polars
    """

    # Ejecuta en DuckDB y devuelve el resultado a Polars
    resultado_pl = duckdb.query(query).pl()
    return resultado_pl

def ejecutar_pipeline_procesamiento(df_crudo: pd.DataFrame) -> pl.DataFrame:
    """Orquestación del procesamiento."""
    df_limpio = procesar_con_polars(df_crudo)
    df_final = transformar_con_duckdb(df_limpio)
    return df_final


Implementación con SQLAlchemy 2.0

In [ ]:
import polars as pl
from sqlalchemy import create_engine, Float, Integer, String
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, Session

# 1. Definición de la Base declarativa (estándar 2.0)
class Base(DeclarativeBase):
    pass

# 2. Modelo ORM para la tabla de vinos procesados
class VinoProcesado(Base):
    __tablename__ = "df_procesado"

    id: Mapped[int] = mapped_column(Integer, primary_key=True, autoincrement=True)
    tipo: Mapped[str] = mapped_column(String(20))
    fixed_acidity: Mapped[float] = mapped_column(Float)
    volatile_acidity: Mapped[float] = mapped_column(Float)
    citric_acid: Mapped[float] = mapped_column(Float)
    residual_sugar: Mapped[float] = mapped_column(Float)
    chlorides: Mapped[float] = mapped_column(Float)
    free_sulfur_dioxide: Mapped[float] = mapped_column(Float)
    total_sulfur_dioxide: Mapped[float] = mapped_column(Float)
    density: Mapped[float] = mapped_column(Float)
    ph: Mapped[float] = mapped_column(Float)
    sulphates: Mapped[float] = mapped_column(Float)
    alcohol: Mapped[float] = mapped_column(Float)
    quality: Mapped[int] = mapped_column(Integer)

    # Columnas derivadas creadas en Polars / DuckDB
    es_buena_calidad: Mapped[int] = mapped_column(Integer)
    promedio_alcohol_por_tipo: Mapped[float] = mapped_column(Float, nullable=True)
    ratio_acidez: Mapped[float] = mapped_column(Float, nullable=True)


# 3. Función para inicializar base y guardar el DataFrame de Polars
def guardar_en_sqlite(df_procesado: pl.DataFrame, db_path: str = "sqlite:///vinos.db"):
    """Crea las tablas e inserta los datos transformados usando SQLAlchemy 2.0."""
    engine = create_engine(db_path, echo=False)

    # Crear las tablas si no existen
    Base.metadata.create_all(engine)

    # Convertimos Polars a lista de diccionarios para inserción masiva
    registros = df_procesado.to_dicts()

    with Session(engine) as session:
        # bulk_insert_mappings es eficiente para pipelines de datos
        session.bulk_insert_mappings(VinoProcesado, registros)
        session.commit()

    print(f"Persistencia exitosa: {len(registros)} filas guardadas en {db_path}.")


Finalización del ML Lab

Modulo de entrenamiento

In [ ]:
import pandas as pd
from sqlalchemy import create_engine
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

def cargar_datos_desde_db(db_path: str = "sqlite:///vinos.db") -> pd.DataFrame:
    """Lee los datos procesados directamente desde la base de datos."""
    engine = create_engine(db_path)
    # CORRECCIÓN: Apuntamos a la tabla correcta 'df_procesado' definida en el ORM
    query = "SELECT * FROM df_procesado"
    df = pd.read_sql(query, con=engine)
    return df

def entrenar_modelo(df: pd.DataFrame):
    """Entrena y optimiza un pipeline de clasificación con GridSearchCV."""
    # 1. Definición de Features (X) y Target (y)
    # Excluimos id y columnas derivadas directas del target
    features_drop = ["id", "quality", "es_buena_calidad"]
    # Nos aseguramos de eliminar columnas si existen en el DataFrame
    existing_drops = [col for col in features_drop if col in df.columns]
    X = df.drop(columns=existing_drops)

    # Codificar variable categórica 'tipo' si no fue numérica
    if "tipo" in X.columns:
        X = pd.get_dummies(X, columns=["tipo"], drop_first=True)
    y = df["es_buena_calidad"]

    # 2. División train/test estratificada
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # 3. Construcción del Pipeline de Scikit-Learn
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", RandomForestClassifier(random_state=42))
    ])

    # 4. Grilla de hiperparámetros para GridSearchCV
    param_grid = {
        "classifier__n_estimators": [50, 100],
        "classifier__max_depth": [None, 10, 20],
        "classifier__min_samples_split": [2, 5]
    }

    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        cv=5,
        scoring="f1",
        n_jobs=-1
    )

    print("Entrenando con GridSearchCV...")
    grid_search.fit(X_train, y_train)

    # 5. Evaluación del mejor modelo
    best_model = grid_search.best_estimator_
    y_pred = best_model.predict(X_test)
    y_proba = best_model.predict_proba(X_test)[:, 1]

    print("\n--- Resultados del Modelo ---")
    print(f"Mejores parámetros: {grid_search.best_params_}")
    print("\nReporte de Clasificación:")
    print(classification_report(y_test, y_pred))
    print(f"ROC-AUC Score: {roc_auc_score(y_test, y_proba):.4f}")
    print("\nMatriz de Confusión:")
    print(confusion_matrix(y_test, y_pred))

    return best_model

if __name__ == "__main__":
    try:
        df_db = cargar_datos_desde_db()
    except Exception as e:
        print("La tabla no existe o la base de datos no está inicializada. Procesando y guardando datos...")
        # Intentamos importar las funciones anteriores para inicializar la BD si están en el namespace
        try:
            # CORRECCIÓN: Cambiamos 'datos' por 'df' que es el DataFrame cargado en celdas anteriores
            df_procesado = ejecutar_pipeline_procesamiento(df)
            guardar_en_sqlite(df_procesado)
            df_db = cargar_datos_desde_db()
        except NameError as ne:
            raise RuntimeError("Por favor, asegúrate de haber ejecutado las celdas de Ingesta, Polars/DuckDB y SQLAlchemy antes de correr esta celda.") from ne

    modelo_final = entrenar_modelo(df_db)

Entrenando con GridSearchCV...

--- Resultados del Modelo ---
Mejores parámetros: {'classifier__max_depth': 20, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 100}

Reporte de Clasificación:
              precision    recall  f1-score   support

           0       0.90      0.98      0.94      1044
           1       0.87      0.54      0.66       256

    accuracy                           0.89      1300
   macro avg       0.88      0.76      0.80      1300
weighted avg       0.89      0.89      0.88      1300

ROC-AUC Score: 0.9279

Matriz de Confusión:
[[1024   20]
 [ 119  137]]


orquestador

In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine
# definimos la función para que apunte a la tabla correcta 'df_procesado' según el ORM
def cargar_datos_desde_db_corregido(db_path: str = "sqlite:///vinos.db") -> pd.DataFrame:
    """Lee los datos procesados directamente desde la tabla correcta en la base de datos."""
    engine = create_engine(db_path)
    query = "SELECT * FROM df_procesado"  # Cambiado de 'vinos_procesados' a 'df_procesado'
    df = pd.read_sql(query, con=engine)
    return df

def main():
    print("=== Iniciando Pipeline de Ciencia de Datos ===")

    # 1. Ingesta
    print("\n[1/4] Extrayendo datos...")
    df_crudo = extraer_dataset_vinos()

    # 2. Procesamiento (Polars + DuckDB)
    print("\n[2/4] Normalizando con Polars y DuckDB...")
    df_procesado = ejecutar_pipeline_procesamiento(df_crudo)

    # 3. Persistencia (SQLAlchemy 2.0 en SQLite)
    db_uri = os.getenv("DB_URI", "sqlite:///vinos.db")
    print(f"\n[3/4] Guardando datos en {db_uri}...")
    guardar_en_sqlite(df_procesado, db_uri)

    # 4. Machine Learning (Scikit-Learn + GridSearchCV)
    print("\n[4/4] Cargando datos y entrenando modelo...")
    df_db = cargar_datos_desde_db_corregido(db_uri)
    entrenar_modelo(df_db)

    print("\n=== Pipeline completado con éxito ===")

if __name__ == "__main__":
    main()

=== Iniciando Pipeline de Ciencia de Datos ===

[1/4] Extrayendo datos...
Dataset crudo guardado en: data/raw/vinos_crudos.csv

[2/4] Normalizando con Polars y DuckDB...

[3/4] Guardando datos en sqlite:///vinos.db...
Persistencia exitosa: 6497 filas guardadas en sqlite:///vinos.db.

[4/4] Cargando datos y entrenando modelo...
Entrenando con GridSearchCV...

--- Resultados del Modelo ---
Mejores parámetros: {'classifier__max_depth': 20, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 100}

Reporte de Clasificación:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99      2088
           1       0.99      0.91      0.94       511

    accuracy                           0.98      2599
   macro avg       0.98      0.95      0.97      2599
weighted avg       0.98      0.98      0.98      2599

ROC-AUC Score: 0.9885

Matriz de Confusión:
[[2082    6]
 [  48  463]]

=== Pipeline completado con éxito ===


Los resultados de mejores parametros muestran la mejor configuracion que encontro el algoritmo para entrebar al modelo.
EN cuanto a la Matris de confusion se ve que casi nunca da alarmas falsas.
El modelo nuestra una precision muy alta, una sensiblidad tambien muy alta,y un nivel de acierto tambien muy alto.